# Team Collaboration Analysis

Simple analysis of team participation patterns for Qolabb.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load data
teams = pd.read_csv('../1_data_collection/team_collaboration/teams.csv')
activities = pd.read_csv('../1_data_collection/team_collaboration/team_activities.csv')

print(f"Teams: {len(teams)}")
print(f"Activity records: {len(activities)}")

## 1. Overview of Team Activities

In [ ]:
# Summary statistics
activities[['hours_logged', 'commits', 'tasks_completed', 'peer_rating']].describe()

## 2. Participation by Contribution Pattern

In [ ]:
# Average hours by contribution pattern
pattern_summary = activities.groupby('contribution_pattern')['hours_logged'].mean()

plt.figure(figsize=(8, 5))
pattern_summary.plot(kind='bar', color=['#e74c3c', '#f39c12', '#27ae60'])
plt.title('Average Hours by Contribution Pattern')
plt.ylabel('Hours per Week')
plt.xlabel('Pattern')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Identifying Low Contributors

In [ ]:
# Find members with low average hours
member_totals = activities.groupby('member_id').agg({
    'hours_logged': 'mean',
    'commits': 'sum',
    'peer_rating': 'mean',
    'team_name': 'first'
}).round(2)

# Flag low contributors (less than 4 hours/week average)
low_contributors = member_totals[member_totals['hours_logged'] < 4]
print(f"Low contributors found: {len(low_contributors)}")
low_contributors.sort_values('hours_logged')

## 4. Team Fairness Score

In [ ]:
# Calculate participation fairness per team using standard deviation
# Lower = more equal participation

team_fairness = activities.groupby(['team_id', 'team_name', 'member_id'])['hours_logged'].mean().reset_index()
fairness_scores = team_fairness.groupby(['team_id', 'team_name'])['hours_logged'].std().reset_index()
fairness_scores.columns = ['team_id', 'team_name', 'participation_inequality']
fairness_scores = fairness_scores.round(2).sort_values('participation_inequality', ascending=False)

print("Teams with most unequal participation:")
fairness_scores.head(10)

## 5. Visualize Team Inequality

In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(fairness_scores['team_name'], fairness_scores['participation_inequality'], color='steelblue')
plt.axhline(y=fairness_scores['participation_inequality'].mean(), color='red', linestyle='--', label='Average')
plt.title('Participation Inequality by Team')
plt.ylabel('Standard Deviation of Hours')
plt.xlabel('Team')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

## Summary

This analysis helps Qolabb by:
1. Identifying **low contributors** who may need intervention
2. Calculating **fairness scores** for each team
3. Highlighting **teams with unequal participation**